In [1]:
!pip install ir_datasets


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Sparse Retriever

In [4]:
from __future__ import annotations
import json, re
from typing import Dict, List, Tuple
from rank_bm25 import BM25Okapi
import ir_datasets
import numpy as np
from tqdm import tqdm

# Choose any BEIR dataset available in ir_datasets
DATASET_NAME = "beir/arguana"   # e.g. "beir/fiqa", "beir/scifact", etc.
TOP_K = 100

def tokenize_simple(text: str):
    return re.findall(r"[a-z0-9]+", text.lower())

In [5]:
# -------------------------------
# 1️⃣ Load dataset dynamically
# -------------------------------
dataset = ir_datasets.load(DATASET_NAME)
print(f"Loaded {DATASET_NAME}")
print(f"Docs: {dataset.docs_count()}, Queries: {dataset.queries_count()}")

Loaded beir/arguana
Docs: 8674, Queries: 1406


In [7]:
# -------------------------------
# 2️⃣ Read corpus and build BM25
# -------------------------------
corpus_docs = list(dataset.docs_iter())
doc_ids = [doc.doc_id for doc in corpus_docs]
tokenized_docs = [tokenize_simple(doc.text) for doc in tqdm(corpus_docs, desc="Tokenizing corpus")]

bm25 = BM25Okapi(tokenized_docs)
print("BM25 index built successfully.")

[INFO] [starting] building docstore
[INFO] [starting] opening zip file                                              
[INFO] [starting] https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/arguana.zip
docs_iter:   0%|                                      | 0/8674 [00:04<?, ?doc/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/arguana.zip: 0.0%| 0.00/3.77M [00:00<?, ?B/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/arguana.zip: 0.4%| 16.4k/3.77M [00:01<05:39, 11.1kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/arguana.zip: 0.9%| 32.8k/3.77M [00:01<03:39, 17.1kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/arguana.zip: 1.3%| 49.2k/3.77M [00:02<02:47, 22.3kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/arguana.zip: 1.7%| 65.5k/3.77M [00:03<03:28, 17.8kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/arguana.zip: 2.2%| 81.9k/3.77M [00:04

BM25 index built successfully.


In [8]:
# -------------------------------
# 3️⃣ Read queries and run BM25 retrieval
# -------------------------------
queries = list(dataset.queries_iter())
qids = [q.query_id for q in queries]
qtexts = [q.text for q in queries]

run_records: List[Tuple[str, str, float]] = []

for qid, qtext in tqdm(zip(qids, qtexts), total=len(qtexts), desc="Retrieving"):
    tokenized_query = tokenize_simple(qtext)
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:TOP_K]
    for rank, idx in enumerate(top_indices, start=1):
        run_records.append((qid, doc_ids[idx], float(scores[idx])))

print(f"Retrieved top-{TOP_K} docs for {len(qids)} queries.")

[INFO] [starting] opening zip file
[INFO] [finished] opening zip file s]
Retrieving: 100%|██████████████████████████████████████████████████████████████████| 1406/1406 [18:59<00:00,  1.23it/s]

Retrieved top-100 docs for 1406 queries.


In [9]:
# -------------------------------
# 4️⃣ Save in TREC format (optional)
# -------------------------------
OUT_PATH = f"./run_bm25_{DATASET_NAME.replace('/', '_')}.trec"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for qid, docid, score in run_records:
        f.write(f"{qid} Q0 {docid} 0 {score:.6f} bm25\n")
print(f"Run file saved at: {OUT_PATH}")

Run file saved at: ./run_bm25_beir_arguana.trec


In [10]:
# -------------------------------
# 5️⃣ Load qrels (for evaluation)
# -------------------------------
qrels = {}
for r in dataset.qrels_iter():
    qrels.setdefault(str(r.query_id), {})[str(r.doc_id)] = int(r.relevance)

[INFO] [starting] opening zip file
[INFO] [finished] opening zip file s]


In [11]:
# -------------------------------
# 6️⃣ Evaluation: MRR@10 and nDCG@10
# -------------------------------
def mrr_at_k(run, qrels, k=10):
    mrrs = []
    for qid, ranked in run.items():
        relset = qrels.get(qid, {})
        rr = 0.0
        for i, (docid, _) in enumerate(ranked[:k], start=1):
            if relset.get(docid, 0) > 0:
                rr = 1.0 / i
                break
        mrrs.append(rr)
    return float(np.mean(mrrs)) if mrrs else 0.0

def ndcg_at_k(run, qrels, k=10):
    import math
    def dcg(rels): return sum((rel / math.log2(i + 2)) for i, rel in enumerate(rels))
    vals = []
    for qid, ranked in run.items():
        rels = [1 if qrels.get(qid, {}).get(docid, 0) > 0 else 0 for docid, _ in ranked[:k]]
        idcg = dcg(sorted(rels, reverse=True))
        v = (dcg(rels)/idcg) if idcg > 0 else 0.0
        vals.append(v)
    return float(np.mean(vals)) if vals else 0.0

# Convert run_records into grouped format
from collections import defaultdict
run_per_q = defaultdict(list)
for qid, docid, score in run_records:
    run_per_q[qid].append((docid, score))
for qid in run_per_q:
    run_per_q[qid].sort(key=lambda x: x[1], reverse=True)

print("Evaluating...")
ndcg = ndcg_at_k(run_per_q, qrels, k=10)
mrr = mrr_at_k(run_per_q, qrels, k=10)
print(f"{DATASET_NAME}: nDCG@10={ndcg:.4f}, MRR@10={mrr:.4f}")

Evaluating...
beir/arguana: nDCG@10=0.2695, MRR@10=0.1745


# Dense

using the same dataset loaded during sparse computation

In [2]:
!pip uninstall -y tensorflow tensorflow-gpu tensorflow-intel keras keras-nightly keras-preprocessing keras-tuner keras-cv keras-nlp tf-keras


Found existing installation: tensorflow 2.17.0
Uninstalling tensorflow-2.17.0:
  Successfully uninstalled tensorflow-2.17.0
Found existing installation: tensorflow-intel 2.17.0
Uninstalling tensorflow-intel-2.17.0:
  Successfully uninstalled tensorflow-intel-2.17.0
Found existing installation: keras 3.12.0
Uninstalling keras-3.12.0:
  Successfully uninstalled keras-3.12.0


You can safely remove it manually.
You can safely remove it manually.


In [3]:
!pip install "transformers==4.39.3" "sentence-transformers==2.7.0" torch "numpy<2.0"


     ---------------------------------------- 0.0/134.8 kB ? eta -:--:--
     ----------- --------------------------- 41.0/134.8 kB 2.0 MB/s eta 0:00:01
     ----------- --------------------------- 41.0/134.8 kB 2.0 MB/s eta 0:00:01
     ------------------------------ ----- 112.6/134.8 kB 939.4 kB/s eta 0:00:01
     -----------------------------------  133.1/134.8 kB 787.7 kB/s eta 0:00:01
     ------------------------------------ 134.8/134.8 kB 666.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/8.8 MB ? eta -:--:--
    --------------------------------------- 0.1/8.8 MB 4.3 MB/s eta 0:00:03
   - -------------------------------------- 0.3/8.8 MB 3.9 MB/s eta 0:00:03
   -- ------------------------------------- 0.5/8.8 MB 4.2 MB/s eta 0:00:02
   --- ------------------------------------ 0.7/8.8 MB 4.3 MB/s eta 0:00:02
   --- ------------------------------------ 0.7/8.8 MB 4.3 MB/s eta 0:00:02
   ---- ----------------------------------- 0.9/8.8 MB 3.7 MB/s eta 0:00:03
  

  You can safely remove it manually.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF_WARNING"] = "1"


In [2]:
from sentence_transformers import SentenceTransformer
print("✅ SentenceTransformer imported successfully")


✅ SentenceTransformer imported successfully


In [3]:
from __future__ import annotations
import ir_datasets
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from tqdm import tqdm
import os

In [6]:
# -------------------------------
# 1️⃣ Config
# -------------------------------

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_DIR = "./"
RUN_NAME = f"run_dense_{DATASET_NAME.replace('/', '_')}.trec"

In [7]:
# -------------------------------
# 2️⃣ Load Dataset (Done already)
# -------------------------------

In [8]:
# -------------------------------
# 3️⃣ Load Corpus and Encode
# -------------------------------
docs = list(dataset.docs_iter())
doc_ids = [d.doc_id for d in docs]
doc_texts = [d.text for d in docs]

print(f"Encoding {len(docs)} documents with model {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)

# Normalize embeddings → cosine similarity with FAISS inner product
doc_embeddings = model.encode(
    doc_texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

dim = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product (cosine if normalized)
index.add(doc_embeddings)
print(f"FAISS index built with {index.ntotal} documents.")

Encoding 8674 documents with model sentence-transformers/all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\mahaj\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mahaj\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

FAISS index built with 8674 documents.


In [9]:
# -------------------------------
# 4️⃣ Encode Queries and Search
# -------------------------------
queries = list(dataset.queries_iter())
qids = [q.query_id for q in queries]
qtexts = [q.text for q in queries]

print(f"Encoding {len(queries)} queries...")
query_embeddings = model.encode(
    qtexts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Searching top-k similar documents...")
D, I = index.search(query_embeddings, TOP_K)  # D = scores, I = indices

Encoding 1406 queries...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Searching top-k similar documents...


In [10]:
# -------------------------------
# 5️⃣ Write Run File (TREC Format)
# -------------------------------
run_path = os.path.join(OUTPUT_DIR, RUN_NAME)
with open(run_path, "w", encoding="utf-8") as f:
    for qi, qid in enumerate(qids):
        pairs = [(doc_ids[idx], float(D[qi, j])) for j, idx in enumerate(I[qi])]
        pairs.sort(key=lambda x: x[1], reverse=True)
        for rank, (docid, score) in enumerate(pairs, start=1):
            f.write(f"{qid} Q0 {docid} {rank} {score:.6f} dense\n")

print(f"✅ Dense run saved to: {run_path}")

✅ Dense run saved to: ./run_dense_beir_arguana.trec


In [11]:
# -------------------------------
# 6️⃣ Optional Evaluation
# -------------------------------
# Load qrels
qrels = {}
for r in dataset.qrels_iter():
    qrels.setdefault(str(r.query_id), {})[str(r.doc_id)] = int(r.relevance)

from collections import defaultdict
def mrr_at_k(run, qrels, k=10):
    mrrs = []
    for qid, ranked in run.items():
        relset = qrels.get(qid, {})
        rr = 0.0
        for i, (docid, _) in enumerate(ranked[:k], start=1):
            if relset.get(docid, 0) > 0:
                rr = 1.0 / i
                break
        mrrs.append(rr)
    return float(np.mean(mrrs)) if mrrs else 0.0

def ndcg_at_k(run, qrels, k=10):
    import math
    def dcg(rels): return sum((rel / math.log2(i + 2)) for i, rel in enumerate(rels))
    vals = []
    for qid, ranked in run.items():
        rels = [1 if qrels.get(qid, {}).get(docid, 0) > 0 else 0 for docid, _ in ranked[:k]]
        idcg = dcg(sorted(rels, reverse=True))
        v = (dcg(rels)/idcg) if idcg > 0 else 0.0
        vals.append(v)
    return float(np.mean(vals)) if vals else 0.0

# Convert run file to dict for eval
run_per_q = defaultdict(list)
with open(run_path, "r", encoding="utf-8") as f:
    for line in f:
        qid, _, docid, _, score, _ = line.strip().split()
        run_per_q[qid].append((docid, float(score)))
for qid in run_per_q:
    run_per_q[qid].sort(key=lambda x: x[1], reverse=True)

print("Evaluating...")
ndcg = ndcg_at_k(run_per_q, qrels, k=10)
mrr = mrr_at_k(run_per_q, qrels, k=10)
print(f"{DATASET_NAME}: nDCG@10={ndcg:.4f}, MRR@10={mrr:.4f}")

Evaluating...
beir/arguana: nDCG@10=0.3715, MRR@10=0.2477
